## Part 1: Setup and Imports

This cell contains everything needed to run this notebook independently.

### 📝 Speaker Notes
**Duration: ~1 minute**

"Welcome! In this section, we're setting up our development environment. We'll import all necessary LangChain libraries and verify our OpenAI API key is configured. This is crucial because agents need an LLM to reason, and we're using OpenAI's GPT-4o-mini model for this tutorial."

In [10]:
# ============================================================
# INDEPENDENT SETUP - All imports and environment config
# ============================================================

import os
import json
import datetime
import re
from typing import Optional
from dotenv import load_dotenv

# LangChain imports
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

# Load environment variables from .env file
load_dotenv()

# Verify API key is set
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("❌ ERROR: Please set OPENAI_API_KEY in your .env file")

print("✅ All imports successful!")
print(f"✅ OpenAI API Key found")
print("\n🎯 Ready to build your first agent!\n")

✅ All imports successful!
✅ OpenAI API Key found

🎯 Ready to build your first agent!



## Part 2: Create Your Tools

These are the "skills" your agent will have. Let's create realistic tools for a personal assistant agent.

### 📝 Speaker Notes
**Duration: ~2-3 minutes**

"Tools are the most important part of an agent - they're what make agents useful! Think of tools as the agent's abilities or actions. We're defining 5 tools:

1. **add_numbers & multiply_numbers** - Basic math operations the agent can use
2. **weather_forecast** - Shows how agents access real-world data (simulated here)
3. **send_email** - Demonstrates agents taking actions
4. **search_knowledge_base** - Shows how agents retrieve information

Each tool has:
- A clear name
- A detailed docstring (HOW THE AGENT UNDERSTANDS WHAT TO DO)
- Input parameters with type hints
- Return types

The docstring is CRITICAL - the LLM reads it to decide WHEN to use this tool."

In [11]:
# Define practical tools for the agent

@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together. Returns the sum."""
    return a + b

@tool
def multiply_numbers(a: float, b: float) -> float:
    """Multiply two numbers together. Returns the product."""
    return a * b

@tool
def weather_forecast(city: str) -> str:
    """Get weather forecast for a city. (Simulated for demo purposes)"""
    # Mock weather data
    forecasts = {
        "new york": "Sunny, 72°F, 10% chance of rain",
        "san francisco": "Cloudy, 65°F, 20% chance of rain",
        "london": "Rainy, 55°F, 80% chance of rain",
        "tokyo": "Clear, 75°F, 5% chance of rain",
    }
    city_lower = city.lower()
    return forecasts.get(city_lower, f"Unable to get forecast for {city} (demo data limited)")

@tool
def send_email(recipient: str, subject: str, message: str) -> str:
    """Send an email. (Simulated for demo purposes)"""
    return f"✅ Email sent to {recipient} with subject '{subject}'. (This is a simulation)"

@tool
def search_knowledge_base(query: str) -> str:
    """Search a knowledge base for information."""
    knowledge = {
        "python": "Python is a high-level programming language known for simplicity.",
        "langchain": "LangChain is a framework for building LLM applications.",
        "ai": "AI (Artificial Intelligence) is the simulation of human intelligence by machines.",
        "agent": "An AI agent is a system that perceives its environment and takes actions to achieve goals.",
    }
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return f"Found: {value}"
    return f"No information found about '{query}' in knowledge base."

# Create tools list
tools = [
    add_numbers,
    multiply_numbers,
    weather_forecast,
    send_email,
    search_knowledge_base
]

print("✅ Tools created successfully!")
print(f"\n📋 Available tools ({len(tools)}):")
for t in tools:
    print(f"  • {t.name}: {t.description}")

✅ Tools created successfully!

📋 Available tools (5):
  • add_numbers: Add two numbers together. Returns the sum.
  • multiply_numbers: Multiply two numbers together. Returns the product.
  • weather_forecast: Get weather forecast for a city. (Simulated for demo purposes)
  • send_email: Send an email. (Simulated for demo purposes)
  • search_knowledge_base: Search a knowledge base for information.


## Part 3: Build Your First Simple Agent

### What is an Agent?
An agent is an AI system that:
1. **Perceives** the user's request
2. **Reasons** about what to do
3. **Decides** which tools to use
4. **Executes** the tools
5. **Learns** from the results

Let's build a simple agent step by step.

### 📝 Speaker Notes
**Duration: ~3 minutes**

"Now comes the magic part - creating the agent. Here's what's happening:

**The LLM (Brain)**: We're using GPT-4o-mini with temperature=0 for deterministic responses. This is the 'brain' that reasons.

**The System Prompt**: Tells the agent how to behave - use available tools to answer questions.

**create_agent()**: LangChain's production-ready function that combines:
- The LLM (for reasoning)
- Tools (for actions)  
- A graph-based runtime (LangGraph)

Instead of manually orchestrating loops, LangChain handles complexity. This is what production systems use!"

In [12]:
# ============================================================
# CREATE AGENT USING LANGCHAIN'S PRODUCTION-READY API
# ============================================================

# Initialize the LLM with ChatOpenAI instance for full control
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=1000
)

# Define the system prompt for the agent
system_prompt = """You are a helpful AI assistant with access to various tools.
When users ask questions:
1. Analyze what they're asking
2. Use the available tools to find accurate information
3. Provide clear, concise final answers
4. Explain what you're doing step by step."""

# Create the agent using LangChain's production-ready create_agent function
# This uses LangGraph under the hood for a robust, graph-based agent runtime
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

print("🤖 Agent created using LangChain's production-ready create_agent API")
print(f"📋 Available tools: {[t.name for t in tools]}")
print(f"🔧 Model: gpt-4o-mini (temperature=0)")
print("\n" + "="*60)

🤖 Agent created using LangChain's production-ready create_agent API
📋 Available tools: ['add_numbers', 'multiply_numbers', 'weather_forecast', 'send_email', 'search_knowledge_base']
🔧 Model: gpt-4o-mini (temperature=0)



## Part 4: Test Your Agent

Let's test the agent with different types of queries!

### 📝 Speaker Notes - Section Overview
**Duration: ~5-6 minutes total for all tests**

"Now we see our agent in action with different scenarios. Each test demonstrates different capabilities. Watch how it decides which tools to use and synthesizes results!"

In [ ]:
# Test 1: Simple math query
print("\n" + "="*60)
print("TEST 1: Math Problem")
print("="*60)

response1 = agent.invoke({
    "messages": [{"role": "user", "content": "What is 15 multiplied by 3?"}]
})

print(f"\n✅ Final Answer: {response1['messages'][-1].content}")
print("\n📝 SPEAKER NOTE: Test 1 shows basic tool usage - agent recognized math question,")
print("   decided to use multiply_numbers tool, executed it, and gave the answer.")


TEST 1: Math Problem

✅ Final Answer: 15 multiplied by 3 is 45.


In [ ]:
# Test 2: Query requiring multiple tools
print("\n" + "="*60)
print("TEST 2: Multi-Tool Query")
print("="*60)

response2 = agent.invoke({
    "messages": [{"role": "user", "content": "What's the weather in London and Tokyo?"}]
})

print(f"\n✅ Final Answer: {response2['messages'][-1].content}")
print("\n📝 SPEAKER NOTE: Test 2 is powerful - agent needed to call weather_forecast")
print("   TWICE (for London and Tokyo), then synthesize both results into one answer.")
print("   This shows agents can break down complex tasks into multiple tool calls!")


TEST 2: Multi-Tool Query

✅ Final Answer: The current weather is as follows:

- **London**: Rainy, 55°F, with an 80% chance of rain.
- **Tokyo**: Clear, 75°F, with a 5% chance of rain.


In [ ]:
# Test 3: Knowledge base query
print("\n" + "="*60)
print("TEST 3: Knowledge Base Query")
print("="*60)

response3 = agent.invoke({
    "messages": [{"role": "user", "content": "Tell me about LangChain and AI agents"}]
})

print(f"\n✅ Final Answer: {response3['messages'][-1].content}")
print("\n📝 SPEAKER NOTE: Test 3 demonstrates information retrieval. Agent uses")
print("   search_knowledge_base to find relevant info, then enriches it with its own")
print("   knowledge. This is how RAG (Retrieval-Augmented Generation) works in production!")


TEST 3: Knowledge Base Query

✅ Final Answer: LangChain is a framework designed for building applications that utilize large language models (LLMs). It provides tools and components that facilitate the development of applications that can leverage the capabilities of LLMs effectively.

### Key Features of LangChain:
1. **Modularity**: LangChain allows developers to create modular applications by combining different components, such as data loaders, prompt templates, and chains of operations.
2. **Integration**: It supports integration with various data sources and APIs, enabling applications to pull in external information and context.
3. **AI Agents**: LangChain can be used to create AI agents that can perform tasks autonomously by making decisions based on the input they receive and the context they have.

### AI Agents:
AI agents in the context of LangChain are systems that can interact with users or other systems, process information, and perform actions based on their programming

In [ ]:
# Test 4: Query that doesn't need tools
print("\n" + "="*60)
print("TEST 4: Simple Conversation (No Tools Needed)")
print("="*60)

response4 = agent.invoke({
    "messages": [{"role": "user", "content": "How would you describe artificial intelligence?"}]
})

print(f"\n✅ Final Answer: {response4['messages'][-1].content}")
print("\n📝 SPEAKER NOTE: Test 4 is CRUCIAL - NOT every question needs tools! Agent")
print("   is smart enough to recognize when it can answer directly from its training.")
print("   This saves API calls and makes the agent efficient. Smart reasoning matters!")


TEST 4: Simple Conversation (No Tools Needed)

✅ Final Answer: Artificial intelligence (AI) refers to the simulation of human intelligence processes by computer systems. These processes include learning (the acquisition of information and rules for using it), reasoning (using rules to reach approximate or definite conclusions), and self-correction. 

AI can be categorized into two main types:

1. **Narrow AI**: This type of AI is designed to perform a specific task, such as facial recognition, language translation, or playing chess. It operates under a limited set of constraints and is the most common form of AI in use today.

2. **General AI**: This is a theoretical form of AI that would have the ability to understand, learn, and apply intelligence across a wide range of tasks, similar to a human being. General AI does not yet exist.

AI technologies include machine learning, natural language processing, robotics, and computer vision, among others. The goal of AI research is to creat

In [ ]:
# Multi-turn conversation with the agent (WITH MEMORY)
print("\n" + "="*60)
print("MULTI-TURN CONVERSATION TEST - With Context Memory")
print("="*60)

# First query
response1 = agent.invoke({
    "messages": [{"role": "user", "content": "What's 50 plus 25?"}]
})
print(f"\n✅ Turn 1: {response1['messages'][-1].content}")

# Second query - pass previous conversation history
response2 = agent.invoke({
    "messages": response1["messages"] + [{"role": "user", "content": "Multiply that result by 2"}]
})
print(f"\n✅ Turn 2: {response2['messages'][-1].content}")

# Third query - pass accumulated conversation history
response3 = agent.invoke({
    "messages": response2["messages"] + [{"role": "user", "content": "Now subtract 50 from that"}]
})
print(f"\n✅ Turn 3: {response3['messages'][-1].content}")

print("\n" + "="*60)
print("💡 Key Point: By passing message history, the agent remembers context!")
print("="*60)

print("\n📝 SPEAKER NOTES FOR THIS SECTION:")
print("   'This is CRITICAL for production agents. Notice what happened:'")
print("   - Turn 1: We ask 50 + 25 → Agent returns 75")
print("   - Turn 2: We say 'Multiply that' → Agent understands 'that' = 75")
print("   - Turn 3: We say 'Subtract 50' → Agent uses result from Turn 2")
print("")
print("   HOW? We pass message history forward. Each turn includes:")
print("   response[messages] + [new user message]")
print("")
print("   This is EXACTLY how ChatGPT and all modern chatbots work!")")


MULTI-TURN CONVERSATION TEST - With Context Memory

✅ Turn 1: 50 plus 25 equals 75.

✅ Turn 2: Multiplying 75 by 2 gives you 150.

✅ Turn 3: Subtracting 50 from 150 results in 100.

💡 Key Point: By passing message history, the agent remembers context!


## Challenge Exercises

### 📝 Speaker Notes
**Duration: ~2 minutes**

"Now that you understand the basics, here are 5 challenges. Try them on your own after this video. These will deepen your knowledge and give you hands-on experience."

### Exercise 1: Add a New Tool
Create a tool that:
- Fetches stock prices (simulated)
- Or calculates compound interest
- Or converts currencies

Then add it to the agent and test it!

**Why this matters**: You'll learn how to extend agent capabilities.

### Exercise 2: Build a Specialized Agent
Create an agent for a specific domain:
- **Travel Agent**: Uses flight search, hotel booking tools
- **Code Assistant**: Uses code compilation, documentation lookup
- **Customer Support**: Uses ticket creation, knowledge base search

**Why this matters**: Different domains need different tools and prompts.

### Exercise 3: Tool Chaining
Create a multi-step task that requires:
1. Tool A produces output
2. Tool B uses that output
3. Tool C uses Tool B's output
4. Get final result

Example: Get date → Check weather → Recommend activity

**Why this matters**: This is the POWER of agents - complex workflows through reasoning.

### Exercise 4: Error Recovery
Test your agent with:
- Invalid tool arguments
- Unavailable tools
- Conflicting requirements

How does it handle errors?

**Why this matters**: Production systems must be robust.

### Exercise 5: Performance Optimization
Measure:
- Response time
- Number of tool calls
- Token usage
- API costs

Can you optimize?

**Why this matters**: Cost and latency matter in production.

## Summary

### 📝 Speaker Notes - Final Wrap Up
**Duration: ~2-3 minutes**

"Congratulations! You've just built a working AI agent from scratch. Let me summarize what we covered today."

You now know how to:
✅ Create tools for agents  
✅ Build a simple agent from scratch  
✅ Connect agents with multiple tools  
✅ Handle agent reasoning and decision-making  
✅ Manage agent memory and conversation history  

### Key Takeaways - Internalize These!
1. **Agents = LLM + Tools + Reasoning Loop** - Each part is essential
2. **Docstrings are EVERYTHING** - The LLM reads tool descriptions to decide when to use them
3. **Message History = Context** - Always pass previous messages forward for memory
4. **LangChain's create_agent()** - Uses LangGraph for production-ready execution
5. **Smart Decisions** - Agents know when to reason vs. when to act (saves API calls!)

### What's Next?
- Explore **LangChain's pre-built agents** for more complex workflows
- Learn about **agent memory types** (short-term, long-term, episodic)
- Implement **multi-agent systems** where agents collaborate
- Deploy agents with **prompt optimization** and **monitoring**
- Build **production-grade agents** with error handling and guardrails

### Resources
- [LangChain Agents Docs](https://python.langchain.com/docs/modules/agents/)
- [ReAct Pattern](https://arxiv.org/abs/2210.03629)
- [Tool Use in LLMs](https://openai.com/blog/function-calling-and-other-api-updates)

Happy Agent Building! 🚀

## Part 6: Understanding Tool Routing

### How Agents Intelligently Route Queries to Tools

This is the MAGIC of agents - how they decide which tool to use without hardcoding!

### 📝 Speaker Notes
**Duration: ~4-5 minutes**

"Now let's understand the most important concept: TOOL ROUTING. This is how agents decide which tool to use.

The key insight: **Agents don't follow hardcoded rules like 'if weather then use weather_forecast'. Instead, they use the LLM's reasoning to decide which tool is best.**

Here's what happens internally:

1. User asks a question
2. LLM reads the question AND all tool descriptions
3. LLM reasons: 'Which tool would help answer this best?'
4. LLM calls the right tool(s)
5. LLM observes the result
6. LLM formulates the final answer

The docstring is CRITICAL - it's how the LLM understands what each tool does. Without a good docstring, the agent won't know when to use the tool!"

### How Tool Routing Works (Step-by-Step)

```
User Query: "What's the weather in London?"
         ↓
    [LLM REASONING PHASE]
    - Reads the user query
    - Scans ALL available tool descriptions:
      ✗ add_numbers - Add two numbers (NOT relevant)
      ✗ multiply_numbers - Multiply numbers (NOT relevant)
      ✓ weather_forecast - Get weather forecast (PERFECT!)
      ✗ send_email - Send email (NOT relevant)
      ✗ search_knowledge_base - Search knowledge (NOT relevant)
         ↓
    Decision: Use weather_forecast(city="london")
         ↓
    [TOOL EXECUTION]
    weather_forecast("london") → Returns: "Rainy, 55°F, 80% chance of rain"
         ↓
    [FINAL ANSWER]
    "The weather in London is rainy with a temperature of 55°F and an 80% chance of rain."
```

### Key Principles of Tool Routing

1. **Tool Descriptions (Docstrings) are EVERYTHING** 
   - The LLM reads docstrings to understand what each tool does
   - Poor docstring = LLM won't know when to use it

2. **No Hardcoding Required**
   - We don't write "if keyword contains 'weather' then use weather_forecast"
   - The LLM intelligently reasons about which tool fits

3. **The LLM is the Router**
   - Language models are incredibly good at semantic understanding
   - They decide routing based on meaning, not keywords

4. **Multiple Tool Routing**
   - If a query needs multiple tools, the LLM calls them in sequence
   - Example: "What's the weather in London and Tokyo?" → Calls weather_forecast twice

5. **Smart Tools-Free Routing**
   - Agents recognize when they DON'T need tools
   - Example: "What is AI?" → Answers directly without tools (saves API calls!)

In [22]:
# Demonstrate Tool Routing with Visual Output
print("\n" + "="*70)
print("TOOL ROUTING DEMONSTRATION - See How Agents Decide")
print("="*70)

# We'll send different queries and show which tools get called

queries = [
    "What is 15 multiplied by 3?",  # Should route to: multiply_numbers
    "What's the weather in London and Tokyo?",  # Should route to: weather_forecast (x2)
    "Tell me about LangChain",  # Should route to: search_knowledge_base
    "How would you describe artificial intelligence?"  # Should route to: NONE (reasoning only)
]

for idx, query in enumerate(queries, 1):
    print(f"\n{'─'*70}")
    print(f"Query {idx}: {query}")
    print(f"{'─'*70}")
    
    # Execute the query
    result = agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    
    # Track which tools were called
    tools_called = []
    for message in result["messages"]:
        if hasattr(message, 'tool_calls') and message.tool_calls:
            for tool_call in message.tool_calls:
                tools_called.append(tool_call['name'])
    
    # Display routing decision
    if tools_called:
        print(f"\n✓ ROUTING DECISION: Tools called:")
        for tool_name in tools_called:
            print(f"  → {tool_name}")
    else:
        print(f"\n✓ ROUTING DECISION: No tools needed (using LLM knowledge)")
    
    # Display final answer
    print(f"\n✓ FINAL ANSWER:")
    print(f"  {result['messages'][-1].content}")

print("\n" + "="*70)
print("📝 SPEAKER NOTE FOR TOOL ROUTING:")
print("="*70)
print("""
Notice how the agent automatically:
1. Query 1 → Routed to multiply_numbers (math question)
2. Query 2 → Routed to weather_forecast twice (weather in 2 cities)
3. Query 3 → Routed to search_knowledge_base (knowledge question)
4. Query 4 → No tools used (general knowledge)

NO HARDCODING! The LLM reads the question AND tool docstrings,
then intelligently decides which tool is best. This is the power of agents!
""")


TOOL ROUTING DEMONSTRATION - See How Agents Decide

──────────────────────────────────────────────────────────────────────
Query 1: What is 15 multiplied by 3?
──────────────────────────────────────────────────────────────────────

✓ ROUTING DECISION: Tools called:
  → multiply_numbers

✓ FINAL ANSWER:
  15 multiplied by 3 is 45.

──────────────────────────────────────────────────────────────────────
Query 2: What's the weather in London and Tokyo?
──────────────────────────────────────────────────────────────────────

✓ ROUTING DECISION: Tools called:
  → weather_forecast
  → weather_forecast

✓ FINAL ANSWER:
  The current weather is as follows:

- **London**: Rainy, 55°F, with an 80% chance of rain.
- **Tokyo**: Clear, 75°F, with a 5% chance of rain.

──────────────────────────────────────────────────────────────────────
Query 3: Tell me about LangChain
──────────────────────────────────────────────────────────────────────

✓ ROUTING DECISION: Tools called:
  → search_knowledge_base

In [23]:
# Advanced: Tool Routing with Detailed Analysis
print("\n" + "="*70)
print("DEEP DIVE: Tool Routing Decision Analysis")
print("="*70)

# Let's analyze a complex query that needs multiple tools
complex_query = "What's the weather in London and send me an email reminder?"
print(f"\nComplex Query: {complex_query}\n")

result = agent.invoke({
    "messages": [{"role": "user", "content": complex_query}]
})

# Detailed analysis of routing decisions
print("📊 TOOL ROUTING ANALYSIS:\n")
routing_sequence = []

for idx, message in enumerate(result["messages"], 1):
    message_type = message.__class__.__name__
    
    if hasattr(message, 'tool_calls') and message.tool_calls:
        print(f"Step {idx} - {message_type} (DECISION)")
        for tool_call in message.tool_calls:
            print(f"  ► Tool: {tool_call['name']}")
            print(f"    Arguments: {tool_call['args']}")
            routing_sequence.append(tool_call['name'])
    
    elif hasattr(message, 'content') and message.content:
        content_preview = message.content[:80] + "..." if len(message.content) > 80 else message.content
        if message_type == "ToolMessage":
            print(f"Step {idx} - {message_type} (OBSERVATION)")
            print(f"  ► Result: {content_preview}")
        else:
            print(f"Step {idx} - {message_type} (REASONING)")
            print(f"  ► Content: {content_preview}")

print(f"\n✓ COMPLETE ROUTING SEQUENCE: {' → '.join(routing_sequence)}")
print(f"\n✓ FINAL ANSWER:\n{result['messages'][-1].content}")

print("\n" + "="*70)
print("KEY INSIGHT ABOUT TOOL ROUTING:")
print("="*70)
print("""
The ReAct Pattern (Reasoning + Acting):

1. THINK: LLM reads the user's request
2. ACT: LLM decides which tools to call based on tool descriptions
3. OBSERVE: Results come back from tools
4. REFLECT: LLM processes the results and decides next steps
5. ANSWER: Final response formulated

This loop repeats until the LLM decides it has all the information needed!
""")


DEEP DIVE: Tool Routing Decision Analysis

Complex Query: What's the weather in London and send me an email reminder?

📊 TOOL ROUTING ANALYSIS:

Step 1 - HumanMessage (REASONING)
  ► Content: What's the weather in London and send me an email reminder?
Step 2 - AIMessage (DECISION)
  ► Tool: weather_forecast
    Arguments: {'city': 'London'}
Step 3 - ToolMessage (OBSERVATION)
  ► Result: Rainy, 55°F, 80% chance of rain
Step 4 - AIMessage (REASONING)
  ► Content: The current weather in London is rainy, with a temperature of 55°F and an 80% ch...

✓ COMPLETE ROUTING SEQUENCE: weather_forecast

✓ FINAL ANSWER:
The current weather in London is rainy, with a temperature of 55°F and an 80% chance of rain.

Now, please provide me with the following details for the email reminder:
1. Your email address
2. The subject of the email
3. The message you want to include in the email

Once I have that information, I can send the email for you.

KEY INSIGHT ABOUT TOOL ROUTING:

The ReAct Pattern (Reas